# Individual Baselines For Sleep Efficiency

Every wearable curve in `outcomes/` is drawn against one of two things: the cohort on that day,
or a single frozen pre-operative reference median per patient. `outcomes/fitbit_sleep.ipynb`
showed why the first is not enough -- the peri-operative dip in that notebook survives only
partly once each night is read against the patient's own usual night, so some of the
cross-sectional fall was who was wearing the device rather than how anyone slept. That leaves the
question this notebook exists for: **a patient's "own usual night" is itself a modelling choice,
and there is more than one defensible way to define it.**

`baselines.py` at the repo root carries four of them -- a frozen pre-operative lookback, a
trailing flat window, a trailing exponential window, and a short window read against a long one --
and this notebook is where they are applied to sleep efficiency and compared. What it settles is
which baseline a recovery metric built on sleep efficiency should use, and what each one costs in
patients and person-days to get there. What it does not settle is the parameters: those were swept
against a cancer-diagnosis index in another project and are inherited here rather than re-derived,
which `baselines.py` says at the top of its own docstring.

Takes as settled: the cohort and index date from `srp_common.cohort_ctes(index_rule="first")`, and
the efficiency definition (`minute_asleep / minute_in_bed` over main sleeps) from
`outcomes/fitbit_sleep.ipynb`. Feeds: whichever outcome notebook ends up constructing a sleep-based
recovery metric.


## Shared Imports and Methods

This section resolves the CDR, builds the cohort SQL, and defines the two helpers every Part below
goes through. There is no analysis in it and nothing here answers a question.

The helpers are the governance boundary as much as a convenience. A person-day grid holds one
column per patient, so every figure and every printed number in this notebook has to be an
aggregate taken over it -- `reportable()` is the only path from a grid to something printable, and
it drops any day contributed by fewer than the threshold's worth of patients before anything sees
it. `plot_series()` then always writes its figure through the suppressing path, so a figure file
cannot claim a suppression that did not happen. **Row-level values never leave the grid and never
leave the workspace.**

The parameters are printed rather than merely imported, because a default that resolves silently
is a default nobody checks. The static window's span, the rolling window's day floor and the
exponential window's effective-sample floor are all derived from the constants in `baselines.py`,
and the print below is what makes the derived numbers visible next to the ones that were typed in.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Walk up from the kernel's cwd to the repo root, so this notebook runs from its task
# directory or from the root without the path being hard-coded in either.
_root = Path.cwd().resolve()
while not (_root / "srp_common.py").exists():
    if _root.parent == _root:
        raise RuntimeError(
            "Repo root not found: no srp_common.py at or above "
            f"{Path.cwd().resolve()}. Run this notebook from inside the repository."
        )
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import baselines as B
import srp_common
from srp_common import CATEGORICAL, INK, INK_MUTED, SERIES, SUPPRESS_THRESHOLD, SURFACE

cfg = srp_common.init("sleep_efficiency")
CDR = cfg.CDR
FIGURES, CACHE_DIR = cfg.FIGURES, cfg.CACHE_DIR
cached_gbq = cfg.cached_gbq

INDEX_RULE = "first"             # unsettled; index_surgery/surgery_day_distribution.ipynb owns it
BASELINE_STATISTIC = B.BASELINE_CENTRE   # "median": efficiency is bounded and mildly skewed
REPORT_FIRST, REPORT_LAST = -30, 30      # the span every figure is drawn over
# How far back the GRID must reach for all four baselines to exist on REPORT_FIRST. The static
# window is anchored to the surgery and the trailing windows to the day they score, so this is
# not simply REPORT_FIRST minus a window length -- baselines.grid_first_day takes both reaches.
GRID_FIRST = B.grid_first_day(REPORT_FIRST)

# Declared before the query runs, so a renamed column fails here with a near-miss suggestion
# rather than as a BigQuery 400 partway through an extract.
cfg.require_columns({
    "sleep_daily_summary": ["person_id", "sleep_date", "is_main_sleep",
                            "minute_in_bed", "minute_asleep"],
})

METRIC_CTE = f"""
    sleep_nights AS (
        -- ONE ROW PER PERSON-DAY, NOT PER SLEEP EPISODE. sleep_daily_summary carries a row per
        -- episode, so a night split across two main-sleep records would pivot into a duplicate
        -- index entry and raise. Summing both sides of the ratio before dividing also gives the
        -- night's true efficiency, which averaging two per-episode ratios would not.
        SELECT
            s.person_id,
            DATE_DIFF(s.sleep_date, i.index_date, DAY) AS rel_day,
            SUM(s.minute_in_bed) AS minute_in_bed,
            SUM(s.minute_asleep) AS minute_asleep
        FROM `{CDR}`.sleep_daily_summary AS s
        JOIN index_surgery AS i ON i.person_id = s.person_id
        -- is_main_sleep is a STRING ('true'/'false') in this CDR, not a BOOL, so a bare
        -- `= true` is a type error; casting first covers a 1/0 encoding too.
        WHERE LOWER(CAST(s.is_main_sleep AS STRING)) IN ('true', '1')
          AND s.minute_asleep IS NOT NULL
          AND s.minute_in_bed IS NOT NULL
          AND DATE_DIFF(s.sleep_date, i.index_date, DAY)
              BETWEEN {GRID_FIRST} AND {REPORT_LAST}
        GROUP BY s.person_id, rel_day
    )
"""
CTES = cfg.cohort_ctes(index_rule=INDEX_RULE, extra=METRIC_CTE)


def reportable(grid):
    """A person-day grid reduced to one row per rel_day, with under-threshold days dropped.

    THIS IS THE ONLY PATH FROM A GRID TO A PRINTABLE NUMBER. The grid is participant-level; what
    comes back is a median, two quartiles and a contributor count per day, and any day fewer than
    SUPPRESS_THRESHOLD patients contributed to is gone before the caller sees it.
    """
    summary = B.daily_summary(grid)
    return summary.loc[summary["n_patients"] >= SUPPRESS_THRESHOLD].reset_index(drop=True)


def plot_series(frames, title, ylabel, save_as, zero_line=False, note="",
                span=(REPORT_FIRST, REPORT_LAST), colours=None):
    """Draw several already-suppressed rel_day curves on one axis and save the figure.

    `frames` is {label: frame from reportable()}. srp_common.plot_rel_day_curve draws one series
    per figure, and the whole point here is reading four baselines against each other, so this is
    the multi-series version of it in the same house style.

    Colours CYCLE rather than zipping against the palette: CATEGORICAL holds four, a figure here
    can carry five, and zipping would have dropped the fifth curve silently instead of repeating
    a colour visibly.

    IT ALWAYS SAVES THROUGH THE SUPPRESSING PATH, SO EVERY FRAME PASSED IN MUST ALREADY HAVE COME
    FROM reportable() -- the '_suppressed' suffix is what .gitignore and the pre-commit hook key
    on, and a figure that carries it without having been filtered is a false claim.
    """
    palette = list(colours) if colours else list(CATEGORICAL)
    fig, ax = plt.subplots(figsize=(10, 4.8), facecolor=SURFACE)
    srp_common.style_axes(ax)
    ax.axvline(0, color=INK_MUTED, linewidth=1, linestyle="--", zorder=2)
    if zero_line:
        ax.axhline(0, color=INK_MUTED, linewidth=1, linestyle=":", zorder=2)

    for i, (label, frame) in enumerate(frames.items()):
        visible = frame.loc[frame["rel_day"].between(*span)]
        colour = palette[i % len(palette)]
        ax.plot(visible["rel_day"], visible["p50"], color=colour, linewidth=2,
                linestyle=":" if label.startswith("observed") else "-",
                label=label, zorder=3)

    ax.set_title(title, fontsize=13, fontweight="600", color=INK, loc="left", pad=12)
    ax.set_xlabel("days relative to index surgery", fontsize=10, color=INK_MUTED)
    ax.set_ylabel(ylabel, fontsize=10, color=INK_MUTED)
    ax.set_xlim(*span)
    ax.legend(frameon=False, fontsize=9, labelcolor=INK_MUTED, ncol=len(frames))
    cfg.save_figure(fig, save_as, True, note)
    return fig


print(f"index rule           : {INDEX_RULE}")
print(f"baseline statistic   : {BASELINE_STATISTIC}")
print(f"reported span        : rel_day {REPORT_FIRST:+d} .. {REPORT_LAST:+d}")
print(f"grid spans           : rel_day {GRID_FIRST:+d} .. {REPORT_LAST:+d}"
      f"   ({REPORT_LAST - GRID_FIRST + 1} days)")
print()
print("the four baselines, at the values inherited from the sweeps")
print(f"  static      : {B.STATIC_LOOKBACK_DAYS}d window ending day "
      f"{-B.STATIC_DAYS_PRIOR:+d}, i.e. rel_day "
      f"{-(B.STATIC_DAYS_PRIOR + B.STATIC_LOOKBACK_DAYS - 1):+d} .. "
      f"{-B.STATIC_DAYS_PRIOR:+d}, needing {B.STATIC_MIN_ELIGIBLE_DAYS} eligible days")
print(f"  rolling     : {B.ROLLING_WINDOW_DAYS}d trailing window, gap "
      f"{B.ROLLING_GAP_DAYS}d, needing {B.ROLLING_MIN_DAYS} of its "
      f"{B.ROLLING_WINDOW_DAYS} days ({1 - B.ROLLING_MAX_MISSING:.0%} of them)")
print(f"  ewma        : halflife {B.EWMA_HALFLIFE_DAYS}d, capacity "
      f"{B.ewma_full_effective_days():.1f} effective days, needing "
      f"{B.EWMA_MIN_EFFECTIVE_DAYS:.1f} of them")
print(f"  comparative : {B.SHORT_SPAN_DAYS}d short vs {B.LONG_SPAN_DAYS}d long, "
      f"{B.SPAN_COUPLING}, in units of the long window's IQR")
print()
print(f"small-cell threshold : {SUPPRESS_THRESHOLD} patients per reported day")


## Part 1 — The Person-Day Grid

Everything below runs on one object: a grid with one row per `rel_day` and one column per patient,
holding that patient's sleep efficiency on the nights the eligibility rule admits and nothing
anywhere else. The grid is built once here and reused, because the expensive part is the extract
and the rolling medians over it, not any individual baseline.

**A grid row is a calendar day, not an observed night.** That is the property the whole module
depends on: a trailing 30-day window has to span the 30 days it claims, and if the rows were
observed nights instead then a patient who wore the device twice in a month would get a "30-day"
window reaching back most of a year. Days a patient recorded nothing are therefore present as
empty cells rather than absent, which is also what makes the eligible-day floors in each baseline
mean what they say.

This is the one Part that holds participant-level values in the kernel. That is allowed -- the
analysis runs inside the Verily workspace and the rows never leave it -- but it is the reason
every number printed from here on comes through `reportable()`.


In [ ]:
# The eligibility rule: which recorded nights are allowed to be a value at all. Each bound is a
# judgement about what the DEVICE was doing, not about what the patient was doing, and every one
# of them removes person-days -- so they are stated here rather than buried in the query.
MIN_MINUTES_IN_BED = 120    # under two hours in bed is a nap misflagged as a main sleep
MIN_MINUTES_ASLEEP = 60     # a main sleep with under an hour of sleep is a device artifact
MIN_EFFICIENCY = 0.20       # below this the record is a device left on a nightstand, not a night
MAX_EFFICIENCY = 1.00       # asleep cannot exceed in-bed; anything over 1 is a summing error

extract_sql = CTES + """
    SELECT person_id, rel_day, minute_in_bed, minute_asleep
    FROM sleep_nights
"""

# Dry run before executing: this is the one query in the notebook that returns person-day rows
# rather than an aggregate, so it is also the one that can scan far more than expected.
cfg.dry_run_bytes(extract_sql)
nights = cached_gbq(extract_sql, "sleep_efficiency_person_days")

# NOTHING BELOW PRINTS A ROW. `nights` and `grid` are participant-level and stay in the kernel;
# every number printed or plotted from here on is a count or a quantile over the whole cohort.
nights["efficiency"] = nights["minute_asleep"] / nights["minute_in_bed"]
nights["eligible"] = (
    nights["minute_in_bed"].ge(MIN_MINUTES_IN_BED)
    & nights["minute_asleep"].ge(MIN_MINUTES_ASLEEP)
    & nights["efficiency"].between(MIN_EFFICIENCY, MAX_EFFICIENCY)
)

grid = B.eligible_value_grid(nights, "efficiency", "eligible", GRID_FIRST, REPORT_LAST)

recorded_nights = len(nights)
eligible_nights = int(nights["eligible"].sum())
patients_recording = nights["person_id"].nunique()
patients_eligible = int(grid.shape[1])

print(f"recorded main-sleep nights in the grid's span : {recorded_nights:>10,}")
print(f"  ... passing the eligibility rule             : {eligible_nights:>10,}"
      f"   ({eligible_nights / recorded_nights:6.1%})")
print(f"patients with any recorded night              : {patients_recording:>10,}")
print(f"  ... with any ELIGIBLE night                  : {patients_eligible:>10,}"
      f"   ({patients_eligible / patients_recording:6.1%})")
print(f"grid                                          : "
      f"{grid.shape[0]:,} days x {grid.shape[1]:,} patients")
print(f"  ... filled cells                             : "
      f"{int(grid.notna().to_numpy().sum()):>10,}"
      f"   ({grid.notna().to_numpy().mean():6.1%} of the grid)")
print()

# Which bound did the dropping. A rule that removes almost nothing is not protecting anything,
# and a rule that removes a tenth of the data needs to be defended rather than inherited.
for label, failed in [
    (f"in bed < {MIN_MINUTES_IN_BED} min", ~nights["minute_in_bed"].ge(MIN_MINUTES_IN_BED)),
    (f"asleep < {MIN_MINUTES_ASLEEP} min", ~nights["minute_asleep"].ge(MIN_MINUTES_ASLEEP)),
    (f"efficiency outside [{MIN_EFFICIENCY:.2f}, {MAX_EFFICIENCY:.2f}]",
     ~nights["efficiency"].between(MIN_EFFICIENCY, MAX_EFFICIENCY)),
]:
    n = int(failed.sum())
    print(f"  nights failing {label:<40s} {n:>9,}   ({n / recorded_nights:6.2%})")

# The contributor count per day, which is the denominator behind every median in this notebook
# and the quantity the small-cell threshold is applied to.
coverage = reportable(grid)
dropped_days = (REPORT_LAST - REPORT_FIRST + 1) - int(
    coverage["rel_day"].between(REPORT_FIRST, REPORT_LAST).sum())

srp_common.plot_rel_day_curve(
    cfg, coverage.loc[coverage["rel_day"].between(GRID_FIRST, REPORT_LAST)], "n_patients",
    title="Patients contributing an eligible sleep-efficiency night, by day relative to surgery",
    ylabel="patients with an eligible night", suppress=True, show_windows=True,
    save_as="grid_coverage")

print()
print(f"reported days with at least {SUPPRESS_THRESHOLD} contributing patients : "
      f"{(REPORT_LAST - REPORT_FIRST + 1) - dropped_days} of "
      f"{REPORT_LAST - REPORT_FIRST + 1}")
for day in [REPORT_FIRST, -14, -7, -1, 1, 7, 14, REPORT_LAST]:
    row = coverage.loc[coverage["rel_day"] == day]
    if row.empty:
        print(f"  day {day:+4d}   below the {SUPPRESS_THRESHOLD}-patient threshold, not reported")
    else:
        print(f"  day {day:+4d}   {int(row['n_patients'].iloc[0]):>7,} patients"
              f"   median efficiency {row['p50'].iloc[0]:.3f}")


## Part 2 — What Each Baseline Calls Normal

A baseline is a claim about where a patient's normal sits, so the first thing to look at is the
claims themselves rather than the departures from them. Drawn as levels, all four are in the
measure's own unit and can share an axis, which is the only figure in this notebook where that is
true.

The four disagree in a way that is structural rather than noisy, and the shape of the disagreement
is the whole argument for choosing between them. The static baseline is one frozen number per
patient taken from before the surgery, so it is a flat line by construction and stays where the
patient was months earlier. The two trailing baselines follow the patient forward, which means
that after surgery they are partly built from post-operative nights and drift down toward whatever
the patient is now doing. **A trailing baseline therefore under-reports the very thing a recovery
metric is trying to measure, and a frozen one cannot adapt to a patient whose normal genuinely
changed.**

The comparative level is the long window's centre, sitting a further short-window length back, so
it is the slowest-moving of the four. What it costs to be that slow is the second cell here: every
baseline drops the person-days it cannot serve, and a rule that halves the analysable cohort has
to buy something with it.


In [ ]:
# Both of these are computed once and reused by Part 3, because the rolling medians over a grid
# this size are the expensive step in the notebook and neither depends on anything below.
levels = B.baseline_levels(grid, statistic=BASELINE_STATISTIC)
deviations = B.deviations_from_each_baseline(grid, statistic=BASELINE_STATISTIC)

# The observed nights themselves, as the reference the four levels are read against. THIS IS THE
# CROSS-SECTIONAL CURVE -- it moves when the set of contributing patients moves, which is the
# problem the baselines exist to remove, so it is drawn as context and not as a result.
observed = reportable(grid)
level_curves = {name: reportable(levels[name]) for name in B.BASELINE_NAMES}

plot_series(
    {"observed night": observed, **{f"{name} baseline": level_curves[name]
                                    for name in B.BASELINE_NAMES}},
    title="Where each baseline puts a patient's normal, against the night actually recorded",
    ylabel="sleep efficiency (minutes asleep / minutes in bed)",
    save_as="baseline_levels", colours=[INK_MUTED, *CATEGORICAL],
    note="median across contributing patients; levels only, not departures")

# The same four on the run-up alone, where the static window ends and the trailing ones are still
# entirely pre-operative -- if they disagree here they disagree about the patient's normal itself
# rather than about how to treat the surgery.
plot_series(
    {"observed night": observed, **{f"{name} baseline": level_curves[name]
                                    for name in B.BASELINE_NAMES}},
    title="The same four baselines through the fortnight either side of surgery",
    ylabel="sleep efficiency (minutes asleep / minutes in bed)",
    save_as="baseline_levels_zoom", span=(-14, 14), colours=[INK_MUTED, *CATEGORICAL],
    note="median across contributing patients; levels only, not departures")

print("median level across contributing patients, by window")
print(f"{'window':<22s} {'observed':>10s}", end="")
for name in B.BASELINE_NAMES:
    print(f" {name:>12s}", end="")
print()
for lo, hi, label in [(-30, -15, "baseline, first half"), (-14, -1, "the fortnight before"),
                      (1, 7, "first post-op week"), (8, 30, "rest of the month")]:
    print(f"{label:<22s}", end="")
    window = observed.loc[observed["rel_day"].between(lo, hi), "p50"]
    print(f" {window.median():>10.3f}" if len(window) else f" {'--':>10s}", end="")
    for name in B.BASELINE_NAMES:
        frame = level_curves[name]
        window = frame.loc[frame["rel_day"].between(lo, hi), "p50"]
        print(f" {window.median():>12.3f}" if len(window) else f" {'--':>12s}", end="")
    print()

# How far each baseline has moved from where it sat before surgery.
#
# THE STATIC ROW IS NOT EXPECTED TO BE EXACTLY ZERO, AND THAT IS THE POINT. Each patient's static
# level is frozen, so any movement in this row is purely the set of contributing patients changing
# underneath a fixed set of values -- the composition effect itself, measured. It therefore bounds
# how much of the movement in the other three rows could be composition rather than anything that
# happened to anybody: a trailing baseline moving by an order more than the static row is moving
# for real reasons, and one moving by about the same amount is not saying anything.
print()
print("movement of the baseline itself, first post-op week vs the fortnight before")
for name in B.BASELINE_NAMES:
    frame = level_curves[name]
    before = frame.loc[frame["rel_day"].between(-14, -1), "p50"].median()
    after = frame.loc[frame["rel_day"].between(1, 7), "p50"].median()
    if pd.isna(before) or pd.isna(after):
        print(f"  {name:<12s} not reportable in one of the two windows")
        continue
    print(f"  {name:<12s} {before:.3f} -> {after:.3f}   ({after - before:+.4f} efficiency)")


#### What Each Baseline Costs In Person-Days And In Patients

Every baseline above refuses to score a person-day it cannot serve: the static one drops patients
without enough pre-operative history, the trailing ones drop days whose window is too empty, and
the comparative one drops days where the long window has no spread to divide by. **A stricter
baseline is not a better one unless what it buys is worth the cohort it costs.**

The counts are cohort-level totals rather than anyone's own values, so the table is reportable as
it stands. Read the patient column first: a baseline that keeps most person-days but few patients
is being carried by a handful of heavy device users.


In [ ]:
cost = B.coverage_by_baseline(grid, deviations)

print(f"eligible person-days in the grid : "
      f"{int(grid.notna().to_numpy().sum()):>10,}")
print(f"patients with any eligible night : {grid.shape[1]:>10,}")
print()
print(f"{'baseline':<14s}{'person-days':>13s}{'of eligible':>13s}"
      f"{'patients':>11s}{'of eligible':>13s}")
for _, row in cost.iterrows():
    # A baseline serving fewer patients than the threshold cannot be reported at all, let alone
    # used -- so the count is checked here rather than only at plotting time.
    flag = "   << below the small-cell threshold" if 0 < row["patients"] < SUPPRESS_THRESHOLD else ""
    print(f"{row['baseline']:<14s}{int(row['person_days']):>13,}"
          f"{row['share_of_person_days']:>12.1%}"
          f"{int(row['patients']):>11,}{row['share_of_patients']:>12.1%}{flag}")

# The floors each of those refusals comes from, printed next to the losses they caused, so the
# cost can be read against the rule rather than against a number nobody can locate.
print()
print("the floor doing the dropping, in each case")
print(f"  static      : {B.STATIC_MIN_ELIGIBLE_DAYS} eligible nights inside rel_day "
      f"{-(B.STATIC_DAYS_PRIOR + B.STATIC_LOOKBACK_DAYS - 1):+d} .. {-B.STATIC_DAYS_PRIOR:+d}"
      f"   ({B.STATIC_MIN_ELIGIBLE_DAYS / B.STATIC_LOOKBACK_DAYS:.0%} of that window)")
print(f"  rolling     : {B.ROLLING_MIN_DAYS} eligible nights in the trailing "
      f"{B.ROLLING_WINDOW_DAYS} days")
print(f"  ewma        : {B.EWMA_MIN_EFFECTIVE_DAYS:.1f} effective days out of a capacity of "
      f"{B.ewma_full_effective_days():.1f}")
print(f"  comparative : {B.days_required(B.SHORT_SPAN_DAYS, B.ROLLING_MAX_MISSING, floor=2)} "
      f"nights in the short window AND "
      f"{B.days_required(B.LONG_SPAN_DAYS, B.ROLLING_MAX_MISSING, floor=2)} in the long one, "
      f"the long one also needing a non-zero IQR")

# How much of the loss is the floor and how much is simply having no history yet. The first
# reported day sits well inside the grid, so a trailing baseline that is still empty there is
# being refused by its floor rather than by the start of the extract.
scored_at_first = {name: int(deviations[name].loc[REPORT_FIRST].notna().sum())
                   for name in B.BASELINE_NAMES}
print()
print(f"patients scored on the first reported day (rel_day {REPORT_FIRST:+d}), out of "
      f"{int(grid.loc[REPORT_FIRST].notna().sum()):,} who recorded an eligible night that day")
for name in B.BASELINE_NAMES:
    n = scored_at_first[name]
    flag = "   not reportable" if n < SUPPRESS_THRESHOLD else ""
    print(f"  {name:<12s} {n:>7,}{flag}")

# The cost is not constant across the span, and the table above cannot show that. A trailing
# baseline is hungriest right after surgery, when the device is least likely to have been worn,
# so the days a recovery metric most needs are the days its baseline is most likely to refuse.
# Drawn against the eligible nights actually recorded, which is the ceiling none of them exceed.
fig, ax = plt.subplots(figsize=(10, 4.8), facecolor=SURFACE)
srp_common.style_axes(ax)
ax.axvline(0, color=INK_MUTED, linewidth=1, linestyle="--", zorder=2)

ceiling = grid.notna().sum(axis=1).loc[REPORT_FIRST:REPORT_LAST]
# SUPPRESSED THE SAME WAY EVERY OTHER FIGURE IS: a day under the threshold is dropped, not drawn
# at its true height, on the count series as much as on the medians.
ax.plot(ceiling.index[ceiling >= SUPPRESS_THRESHOLD],
        ceiling[ceiling >= SUPPRESS_THRESHOLD],
        color=INK_MUTED, linewidth=2, linestyle=":", label="eligible night recorded", zorder=3)

for colour, name in zip(CATEGORICAL, B.BASELINE_NAMES):
    scored = deviations[name].notna().sum(axis=1).loc[REPORT_FIRST:REPORT_LAST]
    visible = scored[scored >= SUPPRESS_THRESHOLD]
    ax.plot(visible.index, visible, color=colour, linewidth=2, label=name, zorder=3)

ax.set_title("Patients each baseline can actually score, by day relative to surgery",
             fontsize=13, fontweight="600", color=INK, loc="left", pad=12)
ax.set_xlabel("days relative to index surgery", fontsize=10, color=INK_MUTED)
ax.set_ylabel("patients scored", fontsize=10, color=INK_MUTED)
ax.set_xlim(REPORT_FIRST, REPORT_LAST)
ax.legend(frameon=False, fontsize=9, labelcolor=INK_MUTED, ncol=5)
cfg.save_figure(fig, "baseline_coverage", True,
                f"days under {SUPPRESS_THRESHOLD} contributing patients omitted")

# The worst day for each baseline inside the reported span, which is what decides whether a
# curve has a hole in it rather than a dip.
print()
print(f"thinnest reported day for each baseline, within rel_day {REPORT_FIRST:+d} .. "
      f"{REPORT_LAST:+d}")
for name in B.BASELINE_NAMES:
    scored = deviations[name].notna().sum(axis=1).loc[REPORT_FIRST:REPORT_LAST]
    worst_day = int(scored.idxmin())
    below = int((scored < SUPPRESS_THRESHOLD).sum())
    print(f"  {name:<12s} {int(scored.min()):>7,} patients on day {worst_day:+d}"
          f"   ({below} of {len(scored)} days below the threshold)")


## Part 3 — Departure From Normal Around Surgery

This is what the baselines are for: how far each night sat from that patient's own normal, which
is the quantity a recovery metric would actually be built on. Differencing against a per-patient
level removes the composition effect that the cross-sectional curve in `outcomes/fitbit_sleep.ipynb`
could not, because a patient can only enter a day's median by contributing a night AND having a
baseline of their own to be read against.

**Three of the four departures share a unit and the fourth does not.** Static, rolling and
exponential are a night's efficiency minus a level, so they are in efficiency points; comparative
is a difference of two window centres divided by a spread, so it is in units of that patient's own
long-window IQR. They are drawn on separate axes for that reason, and a reader who puts them on
one axis will misread the fourth by whatever that patient's IQR happens to be.

The comparative curve also answers a slower question than the other three. Its numerator is the
last fourteen days rather than the night itself, so a short admission cannot appear on it at all,
and its response to surgery is smeared across the fortnight that follows rather than landing on
the days it happened. That is a property of the shape, not a defect to be tuned away.


In [ ]:
UNIT_BASELINES = ("static", "rolling", "ewma")   # the three in efficiency points
ZOOM = (-10, 10)                                 # the admission itself, at readable width

departure_curves = {name: reportable(deviations[name]) for name in B.BASELINE_NAMES}

plot_series(
    {f"{name} baseline": departure_curves[name] for name in UNIT_BASELINES},
    title="How far each night sat from that patient's own normal, by day relative to surgery",
    ylabel="efficiency - that patient's baseline",
    save_as="departures", zero_line=True,
    note="median across contributing patients; zero is the night that patient usually has")

plot_series(
    {f"{name} baseline": departure_curves[name] for name in UNIT_BASELINES},
    title="The same three departures through the ten days either side of surgery",
    ylabel="efficiency - that patient's baseline",
    save_as="departures_zoom", zero_line=True, span=ZOOM,
    note="median across contributing patients; zero is the night that patient usually has")

# Drawn alone because its unit is not the others'. THE Y AXIS HERE IS IQRs OF THAT PATIENT'S OWN
# LONG WINDOW, NOT EFFICIENCY POINTS.
plot_series(
    {"comparative index": departure_curves["comparative"]},
    title="The recent fortnight against the established normal, in that patient's own IQRs",
    ylabel="(short centre - long centre) / long IQR",
    save_as="comparative_index", zero_line=True,
    note="median across contributing patients; unitless, not comparable to the figures above")

print("median departure from baseline, by window")
print(f"{'window':<24s}", end="")
for name in B.BASELINE_NAMES:
    print(f"{name:>14s}", end="")
print()
for lo, hi, label in [(-30, -15, "baseline, first half"), (-14, -1, "the fortnight before"),
                      (1, 3, "first three days"), (4, 7, "rest of week one"),
                      (8, 14, "week two"), (15, 30, "weeks three and four")]:
    print(f"{label:<24s}", end="")
    for name in B.BASELINE_NAMES:
        frame = departure_curves[name]
        window = frame.loc[frame["rel_day"].between(lo, hi), "p50"]
        print(f"{window.median():>+14.4f}" if len(window) else f"{'--':>14s}", end="")
    print()
print("  (the first three columns are efficiency points; comparative is long-window IQRs)")

# Day by day through the admission, with the contributing count beside each value -- a departure
# computed from eight patients is not a smaller finding than one computed from eight hundred, it
# is a different kind of number, and the count is what tells them apart.
print()
print(f"{'day':>5s}{'static':>12s}{'n':>8s}{'rolling':>12s}{'n':>8s}{'ewma':>12s}{'n':>8s}")
for day in range(ZOOM[0], ZOOM[1] + 1):
    print(f"{day:>+5d}", end="")
    for name in UNIT_BASELINES:
        frame = departure_curves[name]
        row = frame.loc[frame["rel_day"] == day]
        if row.empty:
            print(f"{'--':>12s}{'--':>8s}", end="")
        else:
            print(f"{row['p50'].iloc[0]:>+12.4f}{int(row['n_patients'].iloc[0]):>8,}", end="")
    print()


#### Whether The Choice Of Baseline Changes The Ordering Of Days

Two baselines that rank person-days the same way are interchangeable for anything downstream that
only cares about which nights were worse than usual, whatever their levels look like side by side.
Spearman is the right coefficient here because the comparative index is on a different scale
entirely, so only the **ordering** of person-days is comparable across a pair.

Each pair is computed on the person-days both baselines score, not on either one's full set, so
the counts differ from the coverage table above. A pair correlating near 1 means the choice
between them is bookkeeping; a pair correlating near 0 means the choice is the analysis. The
second half of the cell asks the complementary question a correlation cannot answer -- whether the
pair agrees about which side of normal a night fell on, which is what a metric counting bad nights
would actually read.


In [ ]:
agreement = B.baseline_agreement(deviations)

print(f"{'pair':<28s}{'person-days':>14s}{'spearman':>11s}")
for _, row in agreement.iterrows():
    print(f"{row['pair']:<28s}{int(row['person_days']):>14,}{row['spearman']:>11.3f}")

# The same question restricted to the days that matter. Two baselines can agree almost perfectly
# across a year of ordinary nights and still disagree exactly where the surgery is, which is the
# only stretch a recovery metric reads -- so the whole-grid correlation above can be reassuring
# about the wrong thing.
POSTOP = (1, 30)
postop_only = {name: grid_like.loc[POSTOP[0]:POSTOP[1]]
               for name, grid_like in deviations.items()}
postop_agreement = B.baseline_agreement(postop_only)

print()
print(f"the same pairs, restricted to rel_day {POSTOP[0]:+d} .. {POSTOP[1]:+d}")
print(f"{'pair':<28s}{'person-days':>14s}{'spearman':>11s}")
for _, row in postop_agreement.iterrows():
    print(f"{row['pair']:<28s}{int(row['person_days']):>14,}{row['spearman']:>11.3f}")

# What the numbers mean for the choice, stated as a rule rather than left to the reader. The
# thresholds are conventions, so they are named here rather than implied by the prose.
INTERCHANGEABLE = 0.95   # above this, the pair ranks days the same and either will do
DISTINCT = 0.70          # below this, the pair is answering different questions

print()
print("reading, on the post-operative month")
for _, row in postop_agreement.iterrows():
    rho = row["spearman"]
    if pd.isna(rho):
        verdict = "not enough shared person-days to say"
    elif rho >= INTERCHANGEABLE:
        verdict = "interchangeable for anything that only ranks days"
    elif rho >= DISTINCT:
        verdict = "related but not substitutable"
    else:
        verdict = "different questions; the choice between them IS the analysis"
    print(f"  {row['pair']:<28s} {rho:>6.3f}   {verdict}")

# Rank correlation says nothing about WHERE ZERO IS, and zero is the whole claim: "this night was
# worse than that patient's normal". Two baselines can rank a month of nights identically and
# still put a third of them on opposite sides of normal, because one of them sits lower than the
# other throughout. A metric built on "nights below baseline" would then count different nights.
print()
print(f"share of shared person-days the pair puts on the SAME side of normal, rel_day "
      f"{POSTOP[0]:+d} .. {POSTOP[1]:+d}")
print(f"{'pair':<28s}{'person-days':>14s}{'same side':>12s}{'both below':>12s}")
for i, left in enumerate(B.BASELINE_NAMES):
    for right in B.BASELINE_NAMES[i + 1:]:
        a, b = B.flatten_pair(postop_only[left], postop_only[right])
        if len(a) < SUPPRESS_THRESHOLD:
            print(f"{left + ' vs ' + right:<28s}{len(a):>14,}"
                  f"{'--':>12s}{'--':>12s}   too few shared person-days")
            continue
        same = float(((a < 0) == (b < 0)).mean())
        both_below = float(((a < 0) & (b < 0)).mean())
        print(f"{left + ' vs ' + right:<28s}{int(len(a)):>14,}{same:>12.1%}{both_below:>12.1%}")

# And the single number a downstream metric would care about most: how much of the post-operative
# month each baseline calls worse-than-normal at all. A baseline calling 80% of nights abnormal
# is not detecting recovery, it is mis-centred.
print()
print(f"share of scored post-operative person-days each baseline calls below normal")
for name in B.BASELINE_NAMES:
    values = B.flatten(postop_only[name])
    if len(values) < SUPPRESS_THRESHOLD:
        print(f"  {name:<12s} too few scored person-days to report")
        continue
    print(f"  {name:<12s} {float((values < 0).mean()):>6.1%}   of {len(values):,} person-days")
